# Prescribed displacements

The prescribed displacements of `Shell.add_point_pd`,
`Shell.add_distr_pd_fixed_x` and `Shell.add_distr_pd_fixed_y`
are imposed by a `MultiDomain`, exactly with
`pd_method='null-space'` or approximately with `pd_method='penalty'`,
see [Boundary conditions](../boundary_conditions.rst) and [Prescribed displacements imposed exactly](../multidomain_null_space.rst#prescribed-displacements-imposed-exactly). The plates below
have the in-plane displacements free at the edges and are compressed by an
end shortening:

In [1]:
import io
import numpy as np
from numpy.polynomial.legendre import leggauss
from structsolve import Analysis, lb, solve
from structsolve.sparseutils import finalize_symmetric_matrix
from panels.shell import Shell
from panels import json_io
import panels.modelDB as modelDB
from panels.multidomain import MultiDomain
from panels.multidomain.cylinder import create_cylinder
from panels.multidomain.connections import nullspace, fkCpd, fkCld_xcte

In [2]:
# definitions of tests/multidomain/test_null_space_pd.py used below
E = 70e9
NU = 0.3

In [3]:
# from tests/multidomain/test_null_space_pd.py
def plate(a, b, h=2e-3, m=6, n=6, model='plate_clpt_donnell', x0=0., y0=0.,
          stack=(0, ), laminaprop=(E, NU), membrane_free=True):
    s = Shell(a=a, b=b, m=m, n=n, x0=x0, y0=y0, model=model,
              stack=list(stack), plyt=h/len(stack), laminaprop=laminaprop)
    if membrane_free:
        for edge in ('x1', 'x2', 'y1', 'y2'):
            setattr(s, edge + 'u', 1.)
            setattr(s, edge + 'v', 1.)
            #NOTE also the rotations of the shear deformation theories
            setattr(s, edge + 'phix', 1.)
            setattr(s, edge + 'phiy', 1.)
    return s

In [4]:
# from tests/multidomain/test_null_space_pd.py
def end_shortening(p, delta, x1=True, x2=True, cte=False):
    if x1:
        p.add_distr_pd_fixed_x(0., ku=1., funcu=lambda y: 0., cte=True)
    if x2:
        p.add_distr_pd_fixed_x(p.a, ku=1., funcu=lambda y: -delta, cte=cte)

A single shell is wrapped in an assembly without connections, `conn=[]`,
and `MultiDomain.linear_static` returns the configuration with the
prescribed displacements, here the uniform strain of the patch test:

In [5]:
# definitions of tests/multidomain/test_null_space_pd.py used below
def fields(p, c, gridx=7, gridy=5):
    mesh, f = p.uvw(c[p.col_start:p.col_end], gridx=gridx, gridy=gridy)
    return mesh['Xs'] + p.x0, mesh['Ys'] + p.y0, f

In [6]:
import pytest

In [7]:
# from tests/multidomain/test_null_space_pd.py
def test_single_shell_without_conn():
    p = plate(1., 0.6)
    end_shortening(p, 1e-3)
    p.add_point_pd(0., 0., 0, None, 1., 0., 0, None)
    md = MultiDomain([p], conn=[], pd_method='null-space')
    c = md.linear_static()
    assert np.isclose(c @ np.zeros_like(c), 0)
    X, Y, f = fields(p, c)
    assert np.allclose(f['u'], -1e-3*X, atol=1e-15)
    with pytest.raises(RuntimeError, match='No connectivity'):
        MultiDomain([plate(1., 1.)], pd_method='null-space').get_Z()

In [8]:
test_single_shell_without_conn()
print('test_single_shell_without_conn: passed')

test_single_shell_without_conn: passed


A cylinder of five domains under a uniform end shortening gives the same
prebuckling state as the axial load of the corresponding reaction,
`MultiDomain.reaction_resultant`, and a slightly higher critical
load; with the penalty method, the springs of `fkCld_xcte` are added
by the user to the stiffness matrix, and the critical load converges to the
one of the null-space method as the penalty constant increases:

In [9]:
# definitions of tests/multidomain/test_null_space_pd.py used below
LAMINAPROP = (142.5e9, 8.7e9, 0.28, 5.1e9, 5.1e9, 5.1e9)

In [10]:
# from tests/multidomain/test_null_space_pd.py
def cylinder(pd_method, **kw):
    md, conns = create_cylinder(height=0.5, r=0.25, plyt=0.125e-3,
            stack=[0, 45, -45, 90, -45, 45], laminaprop=LAMINAPROP,
            npanels=5, m=8, n=12)
    md = MultiDomain(md.panels, conns, pd_method=pd_method)
    for p in md.panels:
        p.x2u = 1
    return md

In [11]:
# from tests/multidomain/test_null_space_pd.py
def test_cylinder_end_shortening():
    Nxx = -100.
    md = cylinder('null-space')
    for p in md.panels:
        p.add_distr_load_fixed_x(p.a, lambda y: Nxx, None, None)
    eig_N, _ = md.linear_buckling(num_eigvalues=3)
    assert np.isclose(Nxx*eig_N[0], -47162.406, rtol=2e-3)
    c_N = md.linear_static()
    delta = -fields(md.panels[0], c_N)[2]['u'][0, -1]

    md = cylinder('null-space')
    for p in md.panels:
        p.add_distr_pd_fixed_x(p.a, ku=1., funcu=lambda y: -delta, cte=False)
    eig_d, modes = md.linear_buckling(num_eigvalues=3)
    assert modes.shape[1] == 3
    c = md.linear_static()
    assert np.allclose(c, c_N, rtol=0, atol=1e-6*np.abs(c_N).max())
    F = md.reaction_resultant(c, [(p, 'x', p.a) for p in md.panels],
                              NLgeom=False)
    assert np.isclose(F[0], Nxx*2*np.pi*0.25, rtol=1e-6)
    assert 1.02 < eig_d[0]/eig_N[0] < 1.05

    errors = []
    for factor in (1e3, 1e4):
        md = cylinder('penalty')
        size = md.get_size()
        k = factor*md.panels[0].lam.ABD[0, 0]
        kCp = 0.
        for p in md.panels:
            p.add_distr_pd_fixed_x(p.a, ku=k, funcu=lambda y: -delta,
                                   cte=False)
            kCp = kCp + fkCld_xcte(k, 0., 0., p, p.a, size, p.row_start,
                                   p.col_start)
        K = md.calc_kC(silent=True) + finalize_symmetric_matrix(kCp)
        c = md.expand(solve(md.reduce(K), md.reduce(md.calc_fext()),
                            silent=True))
        eig, _ = lb(md.reduce(K), md.reduce(md.calc_kG(c=c, silent=True)),
                    silent=True, num_eigvalues=3)
        errors.append(eig[0]/eig_d[0] - 1)
    assert 0 < errors[1] < 2e-4
    assert 9 < errors[0]/errors[1] < 11

In [12]:
test_cylinder_end_shortening()
print('test_cylinder_end_shortening: passed')

test_cylinder_end_shortening: passed


The geometrically non-linear analyses under displacement control are shown
in [Prescribed displacements imposed exactly](../multidomain_null_space.rst#prescribed-displacements-imposed-exactly). Without an assembly, a point of a `Shell`
is supported by the penalty springs of
`Shell.calc_stiffness_point_constraint`, added to the stiffness
matrix:

In [13]:
from panels.shell import Shell, check_c, load

In [14]:
# definitions of tests/tests_shell/test_shell_api.py used below
laminaprop = (71e9, 0.33)


def _plate(**kwargs):
    kw = dict(a=1., b=0.5, r=None, stack=[0], plyt=2e-3, laminaprop=laminaprop,
              model='plate_clpt_donnell', m=6, n=6)
    kw.update(kwargs)
    return Shell(**kw)

In [15]:
# from tests/tests_shell/test_shell_api.py
def test_stiffness_point_constraint_fixes_the_point():
    from structsolve import solve
    s = _plate(a=1., b=1.)
    s.add_point_load(0.5, 0.5, 0, 0, 1.)
    kC = s.calc_kC()
    x, y = 0.5, 0.5
    c = solve(kC + s.calc_stiffness_point_constraint(x, y, u=False, v=False,
                                                     kuvw=1e12),
              s.calc_fext(), silent=True)
    s.uvw(c, xs=np.array([x]), ys=np.array([y]))
    w_constrained = s.fields['w'][0]
    c = solve(kC, s.calc_fext(), silent=True)
    s.uvw(c, xs=np.array([x]), ys=np.array([y]))
    w_free = s.fields['w'][0]
    assert abs(w_constrained) < 1e-3*abs(w_free)

In [16]:
test_stiffness_point_constraint_fixes_the_point()
print('test_stiffness_point_constraint_fixes_the_point: passed')

test_stiffness_point_constraint_fixes_the_point: passed
